In [0]:
df = spark.read.csv("/Volumes/workspace/bronze1/source_files/Coffe_sales (1).csv", header=True, inferSchema=True)
display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, when

df1 = df.withColumn("transaction_id", monotonically_increasing_id() + 1)
df1 = df1.withColumn(
    "cash_type",
    when(df1.money < 30, "cash")
    .when(df1.money > 35, "upi")
    .otherwise(df1.cash_type)
)
cols = ["transaction_id"] + [col for col in df1.columns if col != "transaction_id"]
df1 = df1.select(*cols)
display(df1)

In [0]:
df1.write.format("delta").mode("overwrite").saveAsTable("bronze1.coffesales")